# 3. プロンプトエンジニアリング


In [1]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

#### 【注意】既知のエラーについて

openai パッケージが依存する httpx のアップデートにより、`openai==1.40.6` を使用する箇所で `TypeError: Client.__init__() got an unexpected keyword argument 'proxies'` というエラーが発生するようになりました。

このエラーは、`!pip install httpx==0.27.2` のように、httpx の特定バージョンをインストールすることで回避することができます。

なお、Google Colab で一度上記のエラーに遭遇したあとで `!pip install httpx==0.27.2` のようにパッケージをインストールし直した場合、以下のどちらかの操作を実施する必要があります。

- Google Colab の「ランタイム」から「セッションを再起動する」を実行する
- 「ランタイムを接続解除して削除」を実行してパッケージのインストールからやり直す


In [2]:
!pip install openai==1.40.6 httpx==0.27.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 361.3/361.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
  Attempting uninstall: openai
    Found existing installation: openai 2.28.0
    Uninstalling openai-2.28.0:
      Successfully uninstalled openai-2.28.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 1.67.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


## 3.2. プロンプトエンジニアリングとは


In [3]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "user", "content": "プロンプトエンジニアリングとは"},
    ],
)
print(response.choices[0].message.content)

プロンプトエンジニアリング（Prompt Engineering）とは、AI（特にChatGPTのような大規模言語モデル：LLM）から**望ましい回答を引き出すために、入力文（プロンプト）を最適化する技術や手法**のことです。

AIは「何をどう指示するか」によって回答の質が劇的に変わります。そのための「コツ」や「体系的な工夫」を指します。

---

### なぜプロンプトエンジニアリングが必要なのか？
AIは「空気を読む」ことは得意ですが、曖昧な指示だと「的外れな回答」や「当たり障りのない一般論」を返してくることがあります。プロンプトエンジニアリングを駆使することで、以下のような効果が得られます。

*   **精度の向上:** 意図通りの専門的な回答を得る。
*   **形式の指定:** 表形式、箇条書き、JSON形式など、使いやすいフォーマットで出力させる。
*   **トーンの調整:** 「親しみやすく」「プロフェッショナルに」など、文体を変える。
*   **ハルシネーション（嘘の回答）の抑制:** 制約条件を設けて、根拠のない情報を出さないようにする。

---

### 代表的なテクニック（例）

プロンプトエンジニアリングにはいくつかの定番手法があります。

1.  **役割を与える（ロールプレイ）**
    *   「あなたはプロのマーケティングコンサルタントです」と定義することで、その立場に基づいた視点で回答させます。
2.  **具体的に指示する（コンテキストの付与）**
    *   「誰に向けた文章か」「どのような目的か」「前提条件は何か」を詳しく書きます。
3.  **ステップ・バイ・ステップ（Chain of Thought）**
    *   「段階的に考えてください」と添えるだけで、AIは論理的な推論を重ねるようになり、計算問題や複雑な課題の正答率が上がります。
4.  **Few-Shotプロンプティング**
    *   質問の前に「例（入力と出力のペア）」をいくつか与えることで、回答のパターンを学習させます。
5.  **制約条件を設ける**
    *   「300文字以内で」「結論から先に述べて」「専門用語は使わずに」といった制限を加えます。

---

### プロンプトエンジニアリングの構成要素
一般的に、効

In [4]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "system", "content": "質問に100文字程度で答えてください。"},
        {"role": "user", "content": "プロンプトエンジニアリングとは"},
    ],
)
print(response.choices[0].message.content)

プロンプトエンジニアリングとは、AIから望ましい回答を引き出すために、指示文（プロンプト）を最適化する技術です。AIの特性を理解し、質問の具体化や制約の付与を行うことで、出力の精度や品質を向上させます。


## 3.3. プロンプトの構成要素の基本


### プロンプトのテンプレート化


In [5]:
prompt = '''\
以下の料理のレシピを考えてください。

料理名: """
{dish}
"""
'''


def generate_recipe(dish: str) -> str:
    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite-preview",
        messages=[
            {"role": "user", "content": prompt.format(dish=dish)},
        ],
    )
    return response.choices[0].message.content


recipe = generate_recipe("カレー")
print(recipe)

カレーライスは家庭ごとにこだわりがある料理ですが、今回は**「飴色玉ねぎの甘みとコクが引き立つ、失敗しない基本のカレー」**のレシピをご紹介します。

市販のルーを使いますが、少しの工夫で格段に美味しくなります。

---

### 【材料】（4〜6人分）
*   **肉（鶏・豚・牛お好みで）：** 300g
*   **玉ねぎ：** 大2個
*   **人参：** 1本
*   **じゃがいも：** 2〜3個
*   **市販のカレールー：** 1/2箱（約100g）
*   **水：** ルーの箱の表示通り（通常700〜800ml）
*   **サラダ油：** 大さじ1
*   **隠し味（重要）：**
    *   ウスターソース：大さじ1
    *   インスタントコーヒー（またはココア）：小さじ1/2
    *   すりおろしにんにく・生姜：各1片分（チューブでも可）

---

### 【作り方】

#### 1. 下準備
*   **玉ねぎ：** 半分に切り、さらに薄切りにします。
*   **人参・じゃがいも：** 一口大の乱切りにします。（じゃがいもは切った後、水にさらすと煮崩れしにくくなります）
*   **肉：** 一口大に切り、塩コショウ（分量外）を少々振っておきます。

#### 2. 飴色玉ねぎを作る（コクの秘訣）
鍋にサラダ油を熱し、玉ねぎを入れます。中火で焦げないように時々混ぜながら、**茶色く色づくまで10〜15分ほどしっかり炒めます。** ここで甘みとコクが生まれます。

#### 3. 具材を炒める
同じ鍋に肉、人参、じゃがいもを加え、肉の色が変わるまで炒めます。ここで「にんにく・生姜」を加え、香りが立つまでさっと炒め合わせます。

#### 4. 煮込む
水を加え、沸騰したらアクを丁寧に取り除きます。弱火〜中火にし、蓋をして野菜が柔らかくなるまで15〜20分ほど煮込みます。

#### 5. ルーを溶かす
一度火を止め、ルーを割り入れます。しっかり溶けたら再び弱火にかけ、混ぜながら5分ほどとろみがつくまで煮込みます。

#### 6. 隠し味を加える
最後に「ウスターソース」と「インスタントコーヒー」を加えます。
*   **ウスターソース：** 旨味と酸味をプラス
*   **インスタントコーヒー：** 苦味が深みとな

In [6]:
prompt = """\
ユーザーが入力した料理のレシピを考えてください。

料理名: '''
{dish}
'''
"""


def generate_recipe(dish: str) -> str:
    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite-preview",
        messages=[
            {"role": "system", "content": "ユーザーが入力した料理のレシピを考えてください。"},
            {"role": "user", "content": f"{dish}"},
        ],
    )
    return response.choices[0].message.content


recipe = generate_recipe("カレー")
print(recipe)

ご家庭で作りやすい、**「基本のコク旨ポークカレー」**のレシピをご紹介します。
隠し味に少し工夫をして、お店のような深い味わいを目指します。

---

### 材料（4〜5人分）
*   **豚肉（バラまたは肩ロース）：** 300g
*   **玉ねぎ：** 中2個
*   **にんじん：** 中1本
*   **じゃがいも：** 中2個
*   **カレールー：** 1/2箱（約100g）
*   **水：** ルーのパッケージの表示通り
*   **サラダ油：** 大さじ1
*   **★隠し味（重要！）**
    *   すりおろしにんにく：1片分（チューブなら3cm）
    *   すりおろししょうが：1片分（チューブなら3cm）
    *   インスタントコーヒー：小さじ1/2（コク出し）
    *   ウスターソース：大さじ1（味の引き締め）

---

### 作り方

1.  **下準備：**
    *   玉ねぎはくし形切り（または薄切り）、にんじんは乱切り、じゃがいもは一口大に切ります。豚肉は食べやすい大きさに切ります。
2.  **炒める：**
    *   鍋にサラダ油を熱し、豚肉を炒めます。色が変わったら玉ねぎを加え、**「飴色」に近くなるまでじっくり**炒めます（ここで玉ねぎの甘みを引き出すのがポイント！）。
    *   にんにく、しょうが、にんじんを加えてさらに炒め合わせます。
3.  **煮込む：**
    *   水を加え、沸騰したらアクを丁寧に取り除きます。
    *   弱火〜中火にし、蓋をして野菜が柔らかくなるまで15〜20分ほど煮込みます。
4.  **仕上げ：**
    *   じゃがいもを加え、さらに柔らかくなるまで5〜10分煮ます。
    *   一度火を止め、ルーを割り入れて溶かします。
    *   ★の「インスタントコーヒー」と「ウスターソース」を加え、再び弱火でとろみがつくまで5分ほど煮込んだら完成です！

---

### 美味しく作るコツ
*   **玉ねぎを焦がさないように炒める：** 時間があれば、玉ねぎを別フライパンで飴色になるまで炒めてから加えると、格段にコクが出ます。
*   **煮込みの最後は弱火で：** ルーを入れた後は焦げやすいので、時々かき混ぜながら弱火で煮

### 出力形式を指定する


In [7]:
system_prompt = """\
ユーザーが入力した料理のレシピを考えてください。

出力は以下のJSON形式にしてください。

```
{
  "材料": ["材料1", "材料2"],
  "手順": ["手順1", "手順2"]
}
```
"""

response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "カレー"},
    ],
)
print(response.choices[0].message.content)

```json
{
  "材料": [
    "肉（豚・牛・鶏など） 200g",
    "玉ねぎ 2個",
    "人参 1本",
    "じゃがいも 2個",
    "カレールー 1/2箱",
    "水 600ml",
    "サラダ油 大さじ1"
  ],
  "手順": [
    "玉ねぎはくし切り、人参とじゃがいもは一口大の乱切りにする。",
    "鍋にサラダ油を熱し、肉を炒め、色が変わったら野菜を加えて炒め合わせる。",
    "水を加え、沸騰したらアクを取り、具材が柔らかくなるまで弱火〜中火で15分ほど煮込む。",
    "一度火を止め、カレールーを割り入れて溶かす。",
    "再び弱火にかけ、時々かき混ぜながらとろみがつくまで5分ほど煮込む。"
  ]
}
```


## 3.4. プロンプトエンジニアリングの定番の手法


### Zero-shot プロンプティング


In [8]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {
            "role": "system",
            "content": "入力をポジティブ・ネガティブ・中立のどれかに分類してください。",
        },
        {
            "role": "user",
            "content": "ChatGPTはとても便利だ",
        },
    ],
)
print(response.choices[0].message.content)

ポジティブ


### Few-shot プロンプティング


In [9]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "system", "content": "入力がAIに関係するか回答してください。"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
)
print(response.choices[0].message.content)

はい、入力された内容はAI（ChatGPT）に関するものです。


In [10]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "system", "content": "入力がAIに関係するか回答してください。"},
        {"role": "user", "content": "AIの進化はすごい"},
        {"role": "assistant", "content": "true"},
        {"role": "user", "content": "今日は良い天気だ"},
        {"role": "assistant", "content": "false"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
)
print(response.choices[0].message.content)

true


### （コラム）Few-shot プロンプティングのその他の形式


In [11]:
prompt = """\
入力がAIに関係するか回答してください。

Q: AIの進化はすごい
A: true
Q: 今日は良い天気だ
A: false
Q: ChatGPTはとても便利だ
A:
"""

# Gemini APIのOpenAI互換エンドポイントはCompletions APIをサポートしていないためエラーとなる
response = client.completions.create(
    model="gemini-3.1-flash-lite-preview",
    prompt=prompt,
)
print(response.choices[0].text)

NotFoundError: Error code: 404

In [14]:
response = client.chat.completions.create(
    model="gemini-3-flash-preview",
    messages=[
        {"role": "system", "content": "入力がAIに関係するか回答してください。"},
        {"role": "system", "name": "example_user", "content": "AIの進化はすごい"},
        {"role": "system", "name": "example_assistant", "content": "true"},
        {"role": "system", "name": "example_user", "content": "今日は良い天気だ"},
        {"role": "system", "name": "example_assistant", "content": "false"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
)
print(response.choices[0].message.content)

ありがとうございます！そう言っていただけてとても嬉しいです。

文章の作成や要約、プログラミングの相談、あるいは日常のちょっとした疑問の解決など、いろいろな場面でお役に立てていれば幸いです。

今日は何か具体的にお手伝いできることはありますか？知りたいことや、やってみたいことがあれば、いつでも気軽に教えてくださいね！


### Zero-shot Chain-of-Thought プロンプティング


In [16]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "system", "content": "回答だけ一言で出力してください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
)
print(response.choices[0].message.content)

8


In [17]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "system", "content": "ステップバイステップで考えてください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
)
print(response.choices[0].message.content)

算数の計算のルール（四則演算の優先順位：掛け算・割り算を先に、足し算・引き算を後に）に従って、ステップバイステップで計算します。

**元の式：10 + 2 * 3 - 4 * 2**

### ステップ1：掛け算を先に計算する
*   2 * 3 = 6
*   4 * 2 = 8

これを元の式に当てはめると、次のようになります。
**10 + 6 - 8**

### ステップ2：左から順番に足し算と引き算をする
*   10 + 6 = 16
*   16 - 8 = 8

**答え：8**
